# LLM-Finetuning mit dataset_llm

Dieses Notebook lädt das vorbereitete Hugging-Face Dataset aus `dataset_llm` und verwendet es für ein einfaches Supervised Fine-Tuning.

Das Training erfolgt mit einem Causal-Language-Model und LoRA-Adaptern. Dadurch werden nicht alle Modellgewichte verändert, sondern nur kleine zusätzliche Adapter trainiert. Das reduziert Speicherbedarf und Rechenaufwand deutlich.

Die Ausgabe wird im Verzeichnis `model_llm_finetuned` gespeichert.


In [ ]:
from pathlib import Path
from datasets import load_from_disk

DATASET_DIR = Path("dataset_llm")
OUTPUT_DIR = Path("model_llm_finetuned")
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

ds = load_from_disk(str(DATASET_DIR))
print(ds)
print(ds[0])

## Trainingsformat vorbereiten

In [ ]:
def format_example(example):
    instruction = example.get("instruction", "").strip()
    user_input = example.get("input", "").strip()
    response = example.get("response", "").strip()

    if user_input:
        text = (
            f"### Anweisung\n{instruction}\n\n"
            f"### Eingabe\n{user_input}\n\n"
            f"### Antwort\n{response}"
        )
    else:
        text = (
            f"### Anweisung\n{instruction}\n\n"
            f"### Antwort\n{response}"
        )

    return {"text": text}

train_ds = ds.map(format_example)
print(train_ds[0]["text"])

## Tokenizer und Modell laden

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

print("Tokenizer und Modell geladen")

## Tokenisierung

In [ ]:
MAX_LENGTH = 512

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_ds = train_ds.map(tokenize_function, batched=False)
tokenized_ds = tokenized_ds.remove_columns([c for c in tokenized_ds.column_names if c not in ["input_ids", "attention_mask", "labels"]])

print(tokenized_ds)
print(tokenized_ds[0].keys())

## LoRA konfigurieren

LoRA (Low-Rank Adaptation) ist eine Methode, um ein grosses Sprachmodell (LLM) effizient zu fine-tunen, ohne alle Modellgewichte zu verändern. Stattdessen werden kleine zusätzliche Matrizen trainiert, die eine niedrigdimensionale Anpassung der bestehenden Gewichte darstellen.

Das Ziel: weniger Rechenleistung, weniger Speicherbedarf und schnelleres Training, während das ursprüngliche Modell weitgehend unverändert bleibt.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## GPU prüfen

Auf einer DGX Spark kann die GPU Unterstützung wie folgt aktiviert werden:

    source ~/.hf/bin/activate
    pip uninstall -y torch torchvision torchaudio
    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

dannach ist das Notebook neu zu starten.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
    print("current device:", torch.cuda.current_device()) 

## Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

trainer.train()

## Modell speichern

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f"Modell gespeichert unter: {OUTPUT_DIR}")

## Einfacher Test

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
)

prompt = "### Anweisung\nWas ist LernMAAS?\n\n### Antwort\n"
result = generator(prompt)
print(result[0]["generated_text"])